[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MeyerBender/segtraq_workshop/blob/main/notebooks/task.ipynb)

# QC for Cell Segmentation with SpatialData and SegTraQ

Assessing the quality of cell segmentation in spatial transcriptomics (ST) data is much trickier than in images. In ST data, we do not only care about accurately capturing cellular morphology, but we also want to avoid contamination from transcripts that diffuse outside of their cell of origin. `SegTraQ` (**Seg**mentation and **Tra**nscript Assignment **Q**uality Assessment) provides metrics to assess exactly this, and can be used to identify spurious samples, cells, or to compare segmentation methods. In this workshop, we will have a look at some CosMx samples taken from [Lermi et al.](https://www.nature.com/articles/s41467-025-63414-1). We will use `spatialdata` to read in the data, `spatialdata-plot` to visualize it, and `SegTraQ` to assess how well segmentation worked for those samples.

In [ ]:
# download the data
# if you have already run this cell once, there is no need to run it again

# use this when running on colab
! wget --no-check-certificate -O /content/data.tar.gz https://oc.embl.de/index.php/s/bBj36ET5S3v5VMW/download
! tar -xzf /content/data.tar.gz
! pip install --quiet segtraq==0.0.7 spatialdata==0.8.0 spatialdata_plot==0.4.2 anndata==0.12.8 seaborn==0.13.2 pandas==2.2.3
data_dir = '/content/data'

# use this when running locally (or on a cluster)
#! wget --no-check-certificate -O data.tar.gz https://oc.embl.de/index.php/s/bBj36ET5S3v5VMW/download
#! tar -xzf data.tar.gz
#data_dir = 'data'

In [ ]:
# importing required packages
import spatialdata as sd
import spatialdata_plot
import segtraq
import pandas as pd
import seaborn as sns
import warnings
import anndata as ad
import matplotlib.pyplot as plt
import os

# setting segtraq to use the maximum amount of cores for parallelization
segtraq.settings.n_jobs = -1

# colors that can be used for plotting later
CELLTYPE_COLORS = {
    "B":               "#929292",  # grey
    "Endothelial":     "#D98C6A",  # terracotta
    "Epithelial":      "#7DAA72",  # green
    "Fibroblast":      "#9A82B8",  # purple
    "ILC":             "#D5A85A",  # ochre
    "Lymphoid.Prolif": "#72AAA6",  # teal
    "Mac.Infiltrated": "#C98DA5",  # rose
    "Mac.PPARG":       "#9DA46F",  # olive
    "Mast":            "#B29A84",  # taupe
    "Mon":             "#829EB5",  # blue-grey
    "Myeloid.Prolif":  "#C87E82",  # muted red
    "NK-like":         "#8FB28A",  # sage
    "Neutrophil":      "#AD98C3",  # lavender
    "Pericyte":        "#D2B875",  # sand
    "Plasma":          "#6F91B5",  # blue
    "T.CD4":           "#B56F76",  # dusty red
    "T.CD8":           "#688F9E",  # slate teal
    "Tgd":             "#A68A64",  # warm brown
    "Treg":            "#7E82A8",  # muted indigo
    "cDC":             "#C49A6C",  # tan
    "pDC":             "#6F9B8C",  # muted sea green
    "Unassigned":      "#D3D3D3",  # light grey
}

# 1. Getting started

Before we can do anything with the data, we need to get it into `spatialdata` format. `Spatialdata-io` provides readers that do this for various technologies. For convenience, the data provided with this workshop is already in `spatialdata` format, so we can simply read it using `sd.read_zarr()`.

In [ ]:
sd_dict = dict()
fovs = [5, 18, 24, 29, 45]
for fov in fovs:
    sd_dict[fov] = sd.read_zarr(os.path.join(data_dir, f'cosmx_fovs/GSM9046088_Lung_Adenocarcinoma_TMA1_CosMx_{fov}.zarr'))

Let's have a look at what these objects look like. We use FOV 45 as an example `sdata` object.

In [ ]:
sdata = sd_dict[45]
sdata

We can use the name of each layer to access it. For example, let's look at the table.

In [ ]:
sdata["table"]

We can also go further and look at the `obs` directly.

In [ ]:
sdata["table"].obs.head()

Let's also quickly have a look at the transcripts and cell boundaries. Explore the `spatialdata` object a bit. How are transcripts and cell boundaries stored?

In [ ]:
# TODO:
# access the transcripts and the cell_boundaries layers

# 2. Plotting the data

Now that we have a feel for the data, we can try to plot some things. To do this, we use `spatialdata-plot`. Let's start by plotting some cells and transcripts.

In [ ]:
sdata.pl.render_shapes().pl.show()

In [ ]:
sdata.pl.render_points().pl.show()

We can also plot them on top of one another. For simplicity, we now only look at a couple of genes.

In [ ]:
(sdata.pl.render_shapes()
      .pl.render_points(color='target', groups=["COL1A1", "IGHG1"], palette=["blue", "gold"])
      .pl.show())

# 3. Loading the data into SegTraQ

One common issue with spatial transcriptomics data is that each vendor calls things differently. For example, some technologies have a `gene` column, others call it `target`. One technology might have `x`, the other one `x-centroid`, and so on. Even within a single `spatialdata` object, there can be inconsistencies. For example, cells are identified via `cell_ID` in the transcripts data frame, but called `cellID` in the cell boundaries.


We do not want to define this every time we compute a metric. To deal with this, `SegTraQ` uses a constructor where you define these parameters once. Try using `segtraq.SegTraQ(sdata)` to load the data. **This will initially raise an error,** because you need to define additional keywords. Give it a try, and see which parameters you need to specify to get it working.

In [ ]:
# TODO: 
# the code below will raise an error
# this is expected, because we first need to set the appropriate arguments
# see if you can figure out how to set them
st = segtraq.SegTraQ(sdata)

As you can see, there are some warnings. These tell you that negative control probes were filtered out, and that there are some cells with 0 counts. We can ignore those for now. Let's now convert all of our `spatialdata` objects into `SegTraQ`.

In [ ]:
st_dict = dict()

for fov, sdata_tmp in sd_dict.items():
    # TODO: apply the constructor from above to store all sdata objects as SegTraQ objects
    # write them into the st_dict

# 4. Computing some metrics
Now that we finally have the data in one consistent container, we can start to compute some metrics! Let's start with some baseline metrics, such as the number of segmented cells or the percentage of unassigned transcripts. We can compute these using `run_baseline()`.

In [ ]:
for st in st_dict.values():
    st.run_baseline()

All results are written into the `spatialdata` object in-place, and can be found in `sdata["table"]`. Let's have a look!

In [ ]:
# TODO:
# have a look what sdata["table"] looks like
# explore what information is stored in the obs, uns, and other layers

Let's put some of this information into a data frame and plot the results.

In [ ]:
df = []

for fov, st in st_dict.items():
    uns = st.sdata["table"].uns
    df.append([fov, uns['num_cells'], uns['perc_unassigned_transcripts'], uns['num_transcripts']])
    
df = pd.DataFrame(df, columns=['fov', 'num_cells', 'perc_unassigned_transcripts', 'num_transcripts'])

In [ ]:
# TODO:
# look at the percentage of unassigned transcripts per cell
# do you spot anything unusual?
# Hint: you could do this by plotting a scatterplot, showing the number of cells vs. the percentage of unassigned transcripts

In [ ]:
# plot the transcripts and cell boundaries from the outlying sample
# what do you observe?

We can also check the number of transcripts for each sample.

In [ ]:
df[['fov', 'num_transcripts']]

We can see that sample 18 has a surprisingly low number of transcripts. Let's check what's going on here.

In [ ]:
# TODO:
# plot the transcripts and cell boundaries for sample 18
# what do you see?

Looks like transcript detection did not work in this sample! In an actual analysis setting, you would remove such a sample before performing downstream analysis.

# 5. Using scRNA-seq information
With the baseline metrics, we were already able to detect various issues with the segmentation and with the data. For the last part of this workshop, we are going to look at some more elaborate metrics that can tell us if a sample shows signs of transcript spillage between cells.

More specifically, we will now use an annotated scRNA-seq dataset to transfer cell type labels onto our spatial transcriptomics data. We then use this RNAseq data to figure out which genes should be expressed in which cell type, and check for contamination this way. The scRNA-seq data is from a publicly available dataset on [CELLxGENE](https://datasets.cellxgene.cziscience.com/a95f7e58-a8fb-4fcd-b55f-e074cdae155b.h5ad).

Note that the `supervised` module is the only module in `SegTraQ` that requires scRNA-seq data. The other five modules are only dependent on the spatial transcriptomics data.

In [ ]:
# reading the data (already in anndata format)
adata_ref = ad.read_h5ad(os.path.join(data_dir, 'lung_adenocarcinoma_scrnaseq.h5ad'))
adata_ref

We can simply use the `run_supervised()` function to run both label transfer and the supervised metrics all in one go. Here, we only show this for one sample, but you could also compute this for all of them.

In [ ]:
st_dict[45].run_supervised(adata_ref=adata_ref, ref_cell_type='author_cell_type_level_1', ref_gene_key='gene_symbol')

Before investigating the results of the metrics, we first visualize the cell types.

In [ ]:
sdata.pl.render_shapes(color='transferred_cell_type', palette=CELLTYPE_COLORS).pl.show()

Let's focus on sample 45 again and have a look at some of the metrics, starting with the contamination strength matrix.

In [ ]:
cont_strength_mat = sdata["table"].uns["contamination_strength_matrix"]
cont_n = sdata["table"].uns["contamination_evaluable_cells_matrix"]

plot_df = (
    cont_strength_mat.stack(dropna=False)
    .rename("contamination_strength")
    .reset_index()
    .rename(columns={"level_0": "source", "level_1": "target"})
)

plot_df["n_evaluable"] = cont_n.stack(dropna=False).values

plt.figure(figsize=(9, 7))

ax = sns.scatterplot(
    data=plot_df,
    x="target",
    y="source",
    size="n_evaluable",
    hue="contamination_strength",
    sizes=(20, 600),
    palette="Reds",
    edgecolor="black",
)

ax.legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    borderaxespad=0,
)

plt.title("Directed Cell-Type Contamination Strength")
plt.xlabel("Target Cell Type")
plt.ylabel("Source Cell Type")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

We can see a lot of contamination between fibroblasts and plasma cells. We can look at the mutually exclusive coexpression rate (MECR) to get a sense for what is going on here. The MECR provides gene pairs that should not occur within the same cell (according to the scRNA-seq reference dataset), but do.

In [ ]:
mecr_df = sdata["table"].uns['mutually_exclusive_coexpression_rate']
# filtering by adjusted p-value and odds ratio
mecr_df = mecr_df[(mecr_df['pvalue_adj'] < 0.05) & (mecr_df['odds_ratio'] > 1)]
# sorting by coexpression fraction
mecr_df = mecr_df.sort_values(by='coexpression_fraction', ascending=False)
mecr_df.head(10)

We can once again visualize the results spatially.

In [ ]:
# TODO:
# pick some genes from the list above, and research in which cell types they should be expressed
# plot those genes and the cells (colored by cell type) using spatialdata-plot

Finally, let's check which cells are particularly affected.

In [ ]:
# TODO:
# render the cell boundaries, and color them by the contamination_strength

# 6. Play around

We have gone through two of `SegTraQ`'s modules here. But there are many more that look at other aspects of segmentation quality, from subcelluar distribution of transcripts to clustering stability. Have a look at the [documentation](https://lazdaria.github.io/SegTraQ/index.html), and feel free to play around with the data!